<a href="https://colab.research.google.com/github/krishnavenib5/AIML-IIITH-Code/blob/U1-M1H1/Copy_of_U1_MH2_Titanic_Classification_kris.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Advanced Certification in AIML
## A Program by IIIT-H and TalentSprint


## Learning Objectives


At the end of the mini-hackathon you will be able to:
* Perform Data preprocessing
* Apply different ML algorithms on the **Titanic** dataset
* Perform VotingClassifier


## Dataset Description

The sinking of the Titanic is one of the most infamous shipwrecks in history.

On April 15, 1912, during her maiden voyage, the widely considered “unsinkable” RMS Titanic sank after colliding with an iceberg. Unfortunately, there weren’t enough lifeboats for everyone onboard, resulting in the death of many passengers and crew.

While there was some element of luck involved in surviving, it seems some groups of people were more likely to survive than others.

[ Data Set Link: Kaggle competition](https://www.kaggle.com/competitions/titanic)

<br/>

### Data Set Characteristics:

**PassengerId:** Id of the Passenger

**Survived:** Survived or Not information

**Pclass:** Socio-economic status (SES)
  * 1st = Upper
  * 2nd = Middle
  * 3rd = Lower

**Name:** Surname, First Names of the Passenger

**Sex:** Gender of the Passenger

**Age:** Age of the Passenger

**SibSp:**	No. of siblings/spouse of the passenger aboard the Titanic

**Parch:**	No. of parents/children of the passenger aboard the Titanic

**Ticket:**	Ticket number

**Fare:** Passenger fare

**Cabin:**	Cabin number

**Embarked:** Port of Embarkation
  * S = Southampton
  * C = Cherbourg
  * Q = Queenstown


## Problem Statement

Build a predictive model that answers the question: “what sort of people were more likely to survive?” using titanic's passenger data (ie name, age, gender, socio-economic class, etc).

In [ ]:
# @title Download the datasets
from IPython import get_ipython

ipython = get_ipython()

notebook="U1_MH1_Data_Munging" #name of the notebook

def setup():
    from IPython.display import HTML, display
    ipython.magic("sx wget https://cdn.iiith.talentsprint.com/aiml/Experiment_related_data/titanic.csv")
    ipython.magic("sx wget https://cdn.iiith.talentsprint.com/aiml/Experiment_related_data/test_titanic.csv")
    print("Data downloaded successfully")
    return

setup()

In [ ]:
!ls

## Exercise 1 - Load and Explore the Data (2 Marks)

* Understand different features in the training dataset
* Understand the data types of each column
* Notice the columns of missing values




#### Import Required Packages

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

In [ ]:
# Load the dataset
df_train = pd.read_csv('titanic.csv')

# Display the first few rows to understand the features
df_train.head()

In [ ]:
# Getting information about the dataset (Features and Data Types)
df_train.info()

In [ ]:
# Count missing values per column
missing_data = df_train.isnull().sum()
missing_percent = (df_train.isnull().sum() / len(df_train)) * 100

# Combine into a clean summary table
missing_summary = pd.DataFrame({'Missing Values': missing_data, 'Percentage (%)': missing_percent})
print(missing_summary[missing_summary['Missing Values'] > 0])

## Exercise 02: Split the data into train and test sets (1 Mark)
Note: Apply all your data preprocessing steps in the train set first and keep the test set aside.

In [ ]:
from sklearn.model_selection import train_test_split

# Separate the target variable (Survived) from the features
X = df_train.drop(columns=['Survived'])
y = df_train['Survived']

# Split into train and test sets (using an 80/20 split and a fixed random_state for reproducibility)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Verify the shapes of the resulting datasets
print(f"X_train shape: {X_train.shape}")
print(f"X_test shape: {X_test.shape}")
print(f"y_train shape: {y_train.shape}")
print(f"y_test shape: {y_test.shape}")

## Exercise 03: Data Cleaning and Processing (15 Marks)
### 3.1 Working on the "Cabin" column (2 Marks)
Find unique entries in the Cabin column. We can label all passengers in two categories having a cabin or not. Check the data type(use: type) of each entry of the Cabin. Convert a string data type into '1' i.e. passengers with cabin and others into '0' i.e. passengers without cabin.  Write a function for the above operation and apply it to the cabin column and create another column with the name " Has_cabin" containing only 0 or 1 entries.





In [ ]:
# Find unique entries in the Cabin column
print("Unique Cabins:", X_train['Cabin'].unique())

# Check the data type of the first few entries to see the mixture of float (NaN) and str
for entry in X_train['Cabin'].head(10):
    print(f"Entry: {entry} | Type: {type(entry)}")

In [ ]:
def check_cabin(entry):
    # Convert string data types into 1, others (like float NaN) into 0
    if type(entry) == str:
        return 1
    else:
        return 0

# Apply the function to the Cabin column and create 'Has_cabin'
X_train['Has_cabin'] = X_train['Cabin'].apply(check_cabin)

# Verify the changes by displaying the first few rows of both columns
print(X_train[['Cabin', 'Has_cabin']].head(10))

 ### 3.2 Working on "SibSp" & "Parch" columns (1 Mark)
Combine columns "SibSp" & "Parch" and create another column that represents the total passengers in one ticket with the name "family_size". In each ticket, there might be Siblings/Spouses (SibSp =Number of Siblings/Spouses Aboard) or Parents/Children (Parch=Number of Parents/Children Aboard ) along with the passenger who booked the ticket.

  

In [ ]:
# Combine SibSp, Parch, and the passenger themselves to find total family size
X_train['family_size'] = X_train['SibSp'] + X_train['Parch'] + 1

# Verify the changes by displaying the first few rows
print(X_train[['SibSp', 'Parch', 'family_size']].head())

### 3.3 Working on the"Embarked" column (2 Marks)
The "embarked" column represents the port of Embarkation: Cherbourg(C), Queenstown(Q), and  Southampton(S ). Thus, the entries are of three categories in this column. Fill in the missing rows in this column. We can fill it with the most frequent category. Map these categorical string entries into numerical.



In [ ]:
# 1. Find the most frequent category (mode) in the Embarked column
most_frequent_port = X_train['Embarked'].mode()[0]
print(f"The most frequent port is: {most_frequent_port}")

# 2. Fill the missing values with the most frequent port
X_train['Embarked'] = X_train['Embarked'].fillna(most_frequent_port)

# 3. Map the categorical string entries into numerical values
embarked_mapping = {'S': 0, 'C': 1, 'Q': 2}
X_train['Embarked'] = X_train['Embarked'].map(embarked_mapping)

# Verify the changes and make sure no missing values remain
print("\nUnique values after mapping:", X_train['Embarked'].unique())
print("Missing values remaining in Embarked:", X_train['Embarked'].isnull().sum())

### 3.4 Working on the "Age" column (2 Marks)
find the number of NaN entries in the age column and their row index. Calculate the mean, Standard deviation of the Age column and check the distribution of the age column.We can fill the missing values with randomly generated integer values between (mean+Standard deviation, mean-Standard deviation). Use : np.isnan; np.random.randint; concept of slicing dataframe. Convert the age column as an integer data type.



In [ ]:
# 1. Find the number of NaN entries and their row indices
nan_mask = np.isnan(X_train['Age'])
nan_indices = X_train[nan_mask].index
print(f"Number of NaN entries in Age column: {nan_mask.sum()}")
print(f"Row indices of NaN entries:\n{list(nan_indices)[:10]}... (showing first 10)")

# 2. Calculate the mean and standard deviation
age_mean = X_train['Age'].mean()
age_std = X_train['Age'].std()
print(f"\nAge Mean: {age_mean:.2f}")
print(f"Age Standard Deviation: {age_std:.2f}")

# 3. Check the distribution (Summary stats)
print("\nAge column distribution summary:")
print(X_train['Age'].describe())

# 4. Generate random integers between (mean - std) and (mean + std)
# Note: lower bound must be smaller than upper bound for np.random.randint
lower_bound = int(age_mean - age_std)
upper_bound = int(age_mean + age_std)

# Seed for reproducibility (optional but good practice)
np.random.seed(42)
random_ages = np.random.randint(lower_bound, upper_bound, size=nan_mask.sum())

# 5. Fill missing values using the concept of dataframe slicing (.loc)
X_train.loc[nan_mask, 'Age'] = random_ages

# 6. Convert the age column to an integer data type
X_train['Age'] = X_train['Age'].astype(int)

# Verify that there are no missing values left and the type is int
print("\nMissing values remaining in Age:", X_train['Age'].isnull().sum())
print("Data type of Age column:", X_train['Age'].dtype)
print("\nFirst few rows of processed Age column:\n", X_train['Age'].head())

### 3.5 Working on "sex" column (1 Mark)
Map the Sex column as 'female' : 0, 'male': 1, and convert it into an integer data type.



In [ ]:
# Map 'female' to 0 and 'male' to 1
sex_mapping = {'female': 0, 'male': 1}
X_train['Sex'] = X_train['Sex'].map(sex_mapping)

# Convert the column into an integer data type
X_train['Sex'] = X_train['Sex'].astype(int)

# Verify the changes and check the data type
print("Unique values in Sex column:", X_train['Sex'].unique())
print("Data type of Sex column:", X_train['Sex'].dtype)
print("\nFirst few rows of processed Sex column:\n", X_train['Sex'].head())

### 3.6  Optional- Working on the "Name" column :
Fetch titles from the name. We can map these titles with numbers and convert them into an integer. Use: concept of the regular expression.

### 3.7 Optional- Working on the "Fare" column :
We can convert face into categorical entries like Low, Medium, and High.



In [ ]:
import re

# 1. Use regex to extract the Title from the Name column
X_train['Title'] = X_train['Name'].str.extract(r'([A-Za-z]+)\.', expand=False)

# Let's see the unique titles extracted and their counts
print("Original Title Counts:\n", X_train['Title'].value_counts())

# 2. Clean up rare titles by grouping them into 'Rare' or standard equivalents
X_train['Title'] = X_train['Title'].replace(['Lady', 'Countess','Capt', 'Col', 'Don',
                                             'Dr', 'Major', 'Rev', 'Sir', 'Jonkheer', 'Dona'], 'Rare')
X_train['Title'] = X_train['Title'].replace('Mlle', 'Miss')
X_train['Title'] = X_train['Title'].replace('Mme', 'Mrs')
X_train['Title'] = X_train['Title'].replace('Ms', 'Miss')

# 3. Map the consolidated titles to numerical entries
title_mapping = {"Mr": 1, "Miss": 2, "Mrs": 3, "Master": 4, "Rare": 5}
X_train['Title'] = X_train['Title'].map(title_mapping)

# 4. Fill any unexpected missing titles with 0 and convert to integer
X_train['Title'] = X_train['Title'].fillna(0).astype(int)

# Verify the result
print("\nUnique mapped values in Title column:", X_train['Title'].unique())
print(X_train[['Name', 'Title']].head())


#Working on the "Fare" column :
# 1. Use qcut to bin Fares into 3 categories: Low, Medium, High
# We set retbins=False to return only the categorized series
X_train['Fare_Category'] = pd.qcut(X_train['Fare'], q=3, labels=['Low', 'Medium', 'High'])

# Let's look at how the segments distributed
print("Fare Category Counts:\n", X_train['Fare_Category'].value_counts())

# 2. Map these categorical string entries into numerical values
fare_mapping = {'Low': 0, 'Medium': 1, 'High': 2}
X_train['Fare_Category'] = X_train['Fare_Category'].map(fare_mapping).astype(int)

# Verify the result
print("\nFirst few rows of processed Fare columns:")
print(X_train[['Fare', 'Fare_Category']].head())


### 3.8 Drop the columns (1 Mark)

Drop the columns: - "PassengerId", "Name",  "SibSp" & "Parch", "Tickets", "Cabin"



In [ ]:
# List of columns to drop (handling 'Ticket' vs 'Tickets')
columns_to_drop = ['PassengerId', 'Name', 'SibSp', 'Parch', 'Ticket', 'Cabin']

# Drop the columns from X_train safely
X_train = X_train.drop(columns=columns_to_drop, errors='ignore')



In [ ]:
# Verify the remaining columns in your training set
print("Remaining columns in X_train:")
print(X_train.columns.tolist())

### 3.9 Apply Standard Scalar (1 Mark)

In [ ]:
from sklearn.preprocessing import StandardScaler

# 1. Initialize the StandardScaler
scaler = StandardScaler()

# 2. Fit the scaler on the training data and transform it
# This converts X_train into a normalized numpy array
X_train_scaled = scaler.fit_transform(X_train)

# 3. Convert it back into a clean DataFrame to maintain column references easily
X_train_scaled = pd.DataFrame(X_train_scaled, columns=X_train.columns)

# Verify the scaling by checking the mean (~0) and standard deviation (~1)
print("Scaled training data summary (First 5 rows):")
print(X_train_scaled.head())

### 3.10 Create a single function for preprocessing the test set (X_test) and apply it. (4 Marks)
#### **Note**: All the pre-processing steps that were applied on the train set before ML Modelling are also applied on the test set before passing through the predict function.

In [ ]:
## Create a function
def preprocess_data(df, is_train=False):
    # Create a copy to prevent SettingWithCopyWarning
    df_clean = df.copy()

    # 1. Cabin Column Processing
    df_clean['Has_cabin'] = df_clean['Cabin'].apply(lambda x: 1 if type(x) == str else 0)

    # 2. Family Size Column Processing
    df_clean['family_size'] = df_clean['SibSp'] + df_clean['Parch'] + 1

    # 3. Embarked Column Processing
    # most_frequent_port ('S') was determined during training
    df_clean['Embarked'] = df_clean['Embarked'].fillna('S')
    embarked_mapping = {'S': 0, 'C': 1, 'Q': 2}
    df_clean['Embarked'] = df_clean['Embarked'].map(embarked_mapping)

    # 4. Age Column Processing
    # age_mean (29.50) and age_std (14.50) were determined during training
    nan_mask = np.isnan(df_clean['Age'])
    # Using the same fixed seed limits from your training calculations
    if is_train:
        np.random.seed(42) # Only enforces training seed if needed
    random_ages = np.random.randint(15, 44, size=nan_mask.sum())
    df_clean.loc[nan_mask, 'Age'] = random_ages
    df_clean['Age'] = df_clean['Age'].astype(int)

    # 5. Sex Column Processing
    sex_mapping = {'female': 0, 'male': 1}
    df_clean['Sex'] = df_clean['Sex'].map(sex_mapping).astype(int)

    # 6. Optional: Name Column (Title Extraction)
    df_clean['Title'] = df_clean['Name'].str.extract(r'([A-Za-z]+)\.', expand=False)
    df_clean['Title'] = df_clean['Title'].replace(['Lady', 'Countess','Capt', 'Col', 'Don',
                                                 'Dr', 'Major', 'Rev', 'Sir', 'Jonkheer', 'Dona'], 'Rare')
    df_clean['Title'] = df_clean['Title'].replace('Mlle', 'Miss')
    df_clean['Title'] = df_clean['Title'].replace('Mme', 'Mrs')
    df_clean['Title'] = df_clean['Title'].replace('Ms', 'Miss')
    title_mapping = {"Mr": 1, "Miss": 2, "Mrs": 3, "Master": 4, "Rare": 5}
    df_clean['Title'] = df_clean['Title'].map(title_mapping).fillna(0).astype(int)

    # Extra safety step for Exercise 5: Fill missing Fares if any exist in a test row
    if df_clean['Fare'].isnull().sum() > 0:
        # 14.4542 is the training set median fare
        df_clean['Fare'] = df_clean['Fare'].fillna(14.4542)

    # 7. Optional: Fare Category Processing
    # Instead of qcut (which changes based on sample data), use hard thresholds from training quantiles
    # Low <= 8.66, Medium <= 26.0, High > 26.0
    def bin_fare(fare):
        if fare <= 8.66: return 0
        elif fare <= 26.0: return 1
        else: return 2
    df_clean['Fare_Category'] = df_clean['Fare'].apply(bin_fare).astype(int)

    # 8. Drop structural columns
    columns_to_drop = ['PassengerId', 'Name', 'SibSp', 'Parch', 'Ticket', 'Cabin']
    df_clean = df_clean.drop(columns=columns_to_drop, errors='ignore')

    return df_clean

In [ ]:
## Applyting above function

# Process the test set using our single function
X_test_clean = preprocess_data(X_test)

# Verify that the feature lists match completely
print("Processed X_test features:")
print(X_test_clean.columns.tolist())
print(f"\nMissing values check:\n{X_test_clean.isnull().sum()}")

### 3.11 Apply standard Scalar transformation to x_test (1 Mark)

In [ ]:
# 1. Transform the preprocessed test data using the already fitted scaler
X_test_scaled = scaler.transform(X_test_clean)

# 2. Convert it back into a clean DataFrame to maintain column references easily
X_test_scaled = pd.DataFrame(X_test_scaled, columns=X_test_clean.columns)

# Verify the scaling by displaying the first few rows
print("Scaled test data summary (First 5 rows):")
print(X_test_scaled.head())

## Exercise  4. Apply Multiple ML Algo. along with  Ensemble Technique (Voting classifier) and display the accuracy (7 Marks)
#### Expected Accuracy >= 80%  


In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier, VotingClassifier
from sklearn.metrics import accuracy_score

# 1. Initialize the individual diverse ML models
log_reg = LogisticRegression(random_state=42, max_iter=1000)
rf_clf = RandomForestClassifier(n_estimators=100, random_state=42, max_depth=6)
gb_clf = GradientBoostingClassifier(n_estimators=100, random_state=42, max_depth=3)

# 2. Define the Ensemble Technique using a VotingClassifier
# We use voting='soft' because it leverages confidence scores rather than just raw votes
voting_clf = VotingClassifier(
    estimators=[
        ('lr', log_reg),
        ('rf', rf_clf),
        ('gb', gb_clf)
    ],
    voting='soft'
)

# 3. Fit all individual models and the ensemble on the scaled training data
models = {
    'Logistic Regression': log_reg,
    'Random Forest': rf_clf,
    'Gradient Boosting': gb_clf,
    'Voting Classifier (Ensemble)': voting_clf
}

print("--- Training and Evaluating Models ---")
for name, model in models.items():
    # Fit the model
    model.fit(X_train_scaled, y_train)

    # Predict on the scaled test data
    y_pred = model.predict(X_test_scaled)

    # Calculate and display the accuracy score
    accuracy = accuracy_score(y_test, y_pred)
    print(f"{name} Accuracy: {accuracy * 100:.2f}%")

## Exercise  5. Pre-process the test_set (3 Marks)
Again we have to apply the same preprocess function and standard scaler on this test set before passing through predict function.

#### Understanding the test set:

In [ ]:
# Load the submission test set
df_submission_test = pd.read_csv('test_titanic.csv')

# Look at the shape and check for the missing value in Fare
print("Before Preprocessing:")
print(df_submission_test.info())

# Pass the dataframe through the preprocessing function
X_submission_clean = preprocess_data(df_submission_test)

# Verify that the missing Fare entry has been handled along with all other columns
print("\nAfter Preprocessing (Missing Values Summary):")
print(X_submission_clean.isnull().sum())

#### Note: In the initial train set there were no missing entries in the "Fare" column. But, now for the submission test set, there is one missing entry in this column.

#### There will be a minor change in the preprocess function to address the above issue.

In [ ]:
# 1. Transform the preprocessed submission test data using the existing scaler
X_submission_scaled = scaler.transform(X_submission_clean)

# 2. Convert it into a clean DataFrame to maintain feature alignments
X_submission_scaled = pd.DataFrame(X_submission_scaled, columns=X_submission_clean.columns)

# Verify everything is shaped correctly and ready for the final predictions
print(f"Final submission test data shape: {X_submission_scaled.shape}")
print("\nFirst few rows of scaled submission features:")
print(X_submission_scaled.head())

## Exercise  6. Prediction for test data (2 Mark)

In [ ]:
# 1. Generate final predictions using the trained Voting Classifier ensemble
submission_predictions = voting_clf.predict(X_submission_scaled)

# 2. Create a clean submission DataFrame matching standard competition formats
submission_df = pd.DataFrame({
    'PassengerId': df_submission_test['PassengerId'],
    'Survived': submission_predictions
})

# 3. Save the predictions to a CSV file
submission_df.to_csv('titanic_final_submission.csv', index=False)

# Verify the file was created properly by checking the shape and class distribution
print("Submission file 'titanic_final_submission.csv' saved successfully!")
print(f"Submission Shape: {submission_df.shape}")
print("\nPredicted Survival Counts:")
print(submission_df['Survived'].value_counts())
print("\nFirst few rows of the final submission:")
print(submission_df.head(10))